# Argument → Rule matcher v2 (validate BEFORE writing to the graph)
Reads the finished Neo4j graph (built by `KG_Construction_v2.ipynb`). **Read-only until the last cell.**

Put in `MyDrive/KG/`: `matcher_v2.py`, `matcher_validation.py`, `Rules_article_19_20_22_v5_fixed.json`, `rules_art21_32_14_v2_fixed.json`.

Flow: **1-5 score** → **6 make labeling sheet** → *(humans label the sheet)* → **7 evaluate thresholds** → **8 write to graph (guarded, only after you pick a threshold)**.
Use a GPU runtime if you have one (cell 4 is faster); CPU also works (a few thousand arguments).

In [ ]:
!pip install -q neo4j sentence-transformers pandas openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, sys
D = '/content/drive/MyDrive'; K = f'{D}/KG'
FILES = ['matcher_v2.py','matcher_validation.py','rules_art19_20_22_v8.json','rules_art21_32_14_v3.json']
for f in FILES:
    ok = os.path.exists(f'{K}/{f}'); print('OK     ' if ok else 'MISSING', f)
    if ok: shutil.copy(f'{K}/{f}', f'/content/{f}')
sys.path.insert(0, '/content')
OUT = f'{D}/matcher_out_v2'; os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
OK      matcher_v2.py
OK      matcher_validation.py
OK      rules_art19_20_22_v8.json
OK      rules_art21_32_14_v3.json


In [ ]:
import getpass, os
os.environ['NEO4J_URI'] = input('Aura URI: ').strip()
USER = input('Aura username (neo4j or the instance id) [neo4j]: ').strip() or 'neo4j'
os.environ['NEO4J_PASSWORD'] = getpass.getpass('Aura password: ')
from neo4j import GraphDatabase
drv = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=(USER, os.environ['NEO4J_PASSWORD'])); drv.verify_connectivity()
with drv.session() as s: print('Connected; cases:', s.run('MATCH (c:Case) RETURN count(c) AS n').single()['n'])

Aura URI: neo4j+s://e600e96a.databases.neo4j.io
Aura username (neo4j or the instance id) [neo4j]: e600e96a
Aura password: ··········
Connected; cases: 9986


## 4 — Load rules, find candidate cases per rule (uses each rule's own Cypher), pull petitioner arguments

In [ ]:
import matcher_v2 as m
rules = m.load_rules([f'/content/{f}' for f in FILES if f.endswith('.json')]); print(len(rules), 'rules')
with drv.session() as s:
    cand = m.get_candidates(s, rules)
    all_cases = set().union(*cand.values())
    args = m.pull_args(s, all_cases)
print({r: len(c) for r, c in list(cand.items())[:5]}, '...')
print('candidate cases:', len(all_cases), '| petitioner arguments to score:', len(args))

36 rules
{'ART19_1A_SPEECH_EXPRESSION_RESTRICTION': 284, 'ART19_1A_ADVOCACY_VS_INCITEMENT': 284, 'ART19_1B_ASSEMBLY_RESTRICTION': 284, 'ART19_1B_UNGUIDED_PERMISSION_REGIME': 284, 'ART19_1C_ASSOCIATION_RESTRICTION': 284} ...
candidate cases: 1705 | petitioner arguments to score: 5265


## 5 — Score (saves to Drive so a disconnect doesn't cost you the run)

In [ ]:
import json, numpy as np
from sentence_transformers import SentenceTransformer
MODEL = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')   # same model as SIMILAR_FACTS
enc = lambda t: MODEL.encode(t, normalize_embeddings=True, convert_to_numpy=True, batch_size=128, show_progress_bar=True)
recs = m.score(rules, cand, args, enc)
json.dump(recs, open(f'{OUT}/scored_pairs.json', 'w'))
arg_text = {a['argument_id']: a['text'] for a in args}; json.dump(arg_text, open(f'{OUT}/arg_text.json', 'w'))
import pandas as pd
df = pd.DataFrame(recs); print(len(df), 'scored (argument, rule) pairs')
print(df.similarity_score.describe().round(3)); print(df.groupby(df.rule_id.str.split('_').str[0]).similarity_score.describe().round(3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

32113 scored (argument, rule) pairs
count    32113.000
mean         0.319
std          0.092
min         -0.041
25%          0.260
50%          0.316
75%          0.373
max          0.796
Name: similarity_score, dtype: float64
          count   mean    std    min    25%    50%    75%    max
rule_id                                                         
ART14    8853.0  0.303  0.079  0.006  0.251  0.303  0.356  0.629
ART19    9955.0  0.313  0.082  0.004  0.260  0.314  0.366  0.688
ART20    1600.0  0.375  0.090  0.056  0.318  0.378  0.438  0.709
ART21    5623.0  0.300  0.083 -0.041  0.246  0.301  0.351  0.680
ART22    3312.0  0.382  0.118  0.069  0.300  0.368  0.458  0.796
ART32    2770.0  0.326  0.098  0.013  0.259  0.320  0.393  0.736


## 6 — Build the blind labeling sheet (~300 rows; first 60 go to BOTH labelers)
Give `labeling_sheet.xlsx` to two people (ideally with legal background). **Keep `labeling_key.csv` private** — it holds the scores.

In [ ]:
import matcher_validation as v
sheet, key = v.make_sheet(recs, arg_text, rules, n=300, n_overlap=60)
sheet.to_excel(f'{OUT}/labeling_sheet.xlsx', index=False); key.to_csv(f'{OUT}/labeling_key.csv', index=False)
print(len(sheet), 'rows ->', OUT); print(key.band.value_counts().sort_index().to_dict())
print('Label: yes / no / ambiguous.  yes = the PETITIONER argues this pattern in favour of their side.')

258 rows -> /content/drive/MyDrive/matcher_out_v2
{0: 60, 1: 60, 2: 60, 3: 60, 4: 18}
Label: yes / no / ambiguous.  yes = the PETITIONER argues this pattern in favour of their side.


In [ ]:
import importlib, matcher_v2 as m; importlib.reload(m)
with drv.session() as s: tiers = m.tier_a_cases(s, rules)
m.add_tier(recs, tiers)
tier_map = {(x['case_id'], x['rule_id']): x['tier'] for x in recs}
key['tier'] = [tier_map[(c, r)] for c, r in zip(key.case_id, key.rule_id)]
key.to_csv(f'{OUT}/labeling_key.csv', index=False)
print(key.tier.value_counts().to_dict())

{'B': 181, 'A': 77}


## 7 — Evaluate (run after labeling). Merge labelers into ONE file: columns pair_id, label (labeler 1), label2 (labeler 2, only for the first 60 rows)

In [ ]:
import pandas as pd, numpy as np
key = pd.read_csv(f'{OUT}/labeling_key.csv'); key['overlap'] = key['overlap'].astype(bool)
labeled = pd.read_excel(f'{OUT}/labeling_sheet_labeled.xlsx')      # <- save the filled sheet under this name in matcher_out/
labeled['label'] = labeled['label'].astype(str).str.strip().str.lower()
if 'label2' in labeled: labeled['label2'] = labeled['label2'].astype(str).str.strip().str.lower()
rep = v.evaluate(key, labeled)
print({k: rep[k] for k in ('agreement','n_labeled','positives','n_ambiguous_dropped') if k in rep})
print(rep['overall'].to_string(index=False))
for f, t in rep['by_family'].items(): print('\n', f); print(t.to_string(index=False))
print(rep['note'])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/matcher_out_v2/labeling_sheet_labeled.xlsx'

## 8 — Write to the graph (GUARDED). Do NOT run until you have chosen a threshold from cell 7 and the team agrees.
`WRITE = False` by default. First it prints how many edges each threshold would create (budget: stay well under the Aura relationship cap).

In [ ]:
THRESHOLD = None     # <- set after cell 7, e.g. 0.55
WRITE = False        # <- set True to write
TOP_N = 2            # best N arguments per (case, rule)
for t in (0.45, 0.50, 0.55, 0.60, 0.65, 0.70):
    print(f'threshold {t}: {len(m.select_edges(recs, t, TOP_N))} edges')
if WRITE and THRESHOLD is not None:
    edges = m.select_edges(recs, THRESHOLD, TOP_N)
    with drv.session() as s: m.write_graph(s, rules, edges, THRESHOLD)
    print('Wrote', len(edges), 'MATCHES_RULE edges at threshold', THRESHOLD)
else:
    print('Not writing (set THRESHOLD and WRITE=True).')